# Notebook 02 — Linear Regression from Scratch

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 02 of 15  
**Time estimate:** 90 minutes

> Linear regression is the foundation of everything. The normal equations,
> the geometric interpretation, and the gradient — understanding these lets you
> derive and debug every other supervised learning method.

---
## Step 1 — Motivation

Gene expression is often modeled as a linear function of transcription factor binding,
drug concentration, or genetic variants. Drug dose-response curves, eQTL effect sizes,
protein abundance from mRNA — all start with linear regression. More importantly,
understanding ordinary least squares (OLS) analytically prepares you for logistic
regression, GLMs, and neural networks.

---
## Step 2 — Intuition

**Linear regression:** Find a hyperplane $\hat{y} = \mathbf{w}^T \mathbf{x} + b$ that
minimizes the sum of squared residuals between predictions and targets.

**Geometric view:** Project the target vector $\mathbf{y}$ onto the column space of
$X$ (the design matrix). The OLS solution is that projection — the residual
$\mathbf{y} - X\mathbf{w}$ is perpendicular to every column of $X$.

**Regularization intuition:**
- **Ridge (L2):** Shrinks all coefficients toward 0 by adding a penalty on $\|\mathbf{w}\|_2^2$.
  Useful when many features each contribute a little (polygenic traits).
- **Lasso (L1):** Shrinks and selects — drives some coefficients exactly to 0.
  Useful when only a few features matter (sparse genetic architecture).

---
## Step 3 — Biological Background

**eQTL mapping:** Association between genetic variant (SNP) and gene expression.
Each SNP is a feature; expression level is the target. Linear regression with the
SNP genotype as predictor (additive model: 0/1/2 copies of effect allele).

**Drug dose-response:** Log-dose vs. response is often linear over the active range.
The slope (potency) and intercept (baseline) are the OLS coefficients.

**Lasso for genomics:** When p >> n (50,000 SNPs, 200 samples), Lasso performs
automatic feature selection — returns only the k strongest eQTLs. This is directly
analogous to forward stepwise selection but computationally efficient.

**Assumptions of OLS (need to state these in interviews):**
1. Linearity: $\mathbb{E}[y | \mathbf{x}] = \mathbf{w}^T \mathbf{x}$
2. Independence: observations are i.i.d.
3. Homoscedasticity: $\text{Var}(\epsilon) = \sigma^2$ (constant)
4. Normality of residuals: $\epsilon \sim \mathcal{N}(0, \sigma^2)$ (needed for inference, not prediction)

RNA-seq count data violates assumptions 1 (non-negative) and 3 (variance ∝ mean)
— use DESeq2 / edgeR negative binomial GLMs for differential expression, not OLS.

---
## Step 4 — Mathematical Explanation

**OLS loss:**
$$\mathcal{L}(\mathbf{w}) = \|\mathbf{y} - X\mathbf{w}\|_2^2 = (\mathbf{y} - X\mathbf{w})^T(\mathbf{y} - X\mathbf{w})$$

**Normal equations** (set gradient to zero):
$$\nabla_\mathbf{w} \mathcal{L} = -2X^T(\mathbf{y} - X\mathbf{w}) = 0 \implies X^T X \mathbf{w} = X^T \mathbf{y}$$
$$\boxed{\mathbf{w}^* = (X^T X)^{-1} X^T \mathbf{y}}$$

$(X^TX)^{-1}X^T$ is the **Moore-Penrose pseudoinverse** of $X$.
Solvable analytically in $O(np^2 + p^3)$; not for $p \gg n$.

**Gradient descent update:**
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \nabla_\mathbf{w} \mathcal{L} = \mathbf{w}_t + \frac{2\eta}{n} X^T(\mathbf{y} - X\mathbf{w}_t)$$

**Ridge regression:**
$$\mathbf{w}^* = (X^TX + \lambda I)^{-1} X^T \mathbf{y}$$
The $\lambda I$ ensures the matrix is always invertible — Ridge is always solvable.

**Lasso:** No closed form; uses coordinate descent or ISTA (iterative soft-thresholding).
Soft-thresholding operator: $S_\lambda(w) = \text{sign}(w) \max(|w| - \lambda, 0)$.

In [ ]:
# Step 6 — Python: Linear regression from scratch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler

# ---- Generate synthetic dose-response data ----
rng = np.random.default_rng(42)
n = 100
X = rng.uniform(0, 5, (n, 3))  # 3 features: log-dose, time, cell_line_effect
true_w = np.array([2.0, -0.5, 0.8])
true_b = 1.0
y = X @ true_w + true_b + rng.normal(0, 0.5, n)

# Add bias column
X_b = np.column_stack([np.ones(n), X])

# ---- 1. OLS via normal equations ----
def ols_normal_equations(X_b, y):
    """w = (X^T X)^{-1} X^T y, returns full weight vector [bias, w1, w2, ...]"""
    return np.linalg.solve(X_b.T @ X_b, X_b.T @ y)

w_ols = ols_normal_equations(X_b, y)
print(f'OLS (normal equations):')
print(f'  Bias: {w_ols[0]:.4f} (true: {true_b})')
print(f'  Weights: {w_ols[1:].round(4)} (true: {true_w})')
print(f'  MSE: {np.mean((X_b @ w_ols - y)**2):.6f}')

# ---- 2. Gradient descent ----
def gradient_descent_linear(X_b, y, lr=0.01, n_iter=2000):
    n, p = X_b.shape
    w = np.zeros(p)
    losses = []
    for i in range(n_iter):
        residual = y - X_b @ w
        grad = -2 / n * X_b.T @ residual
        w -= lr * grad
        losses.append(np.mean(residual**2))
    return w, losses

# Standardize features for stable gradient descent
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
X_sc_b = np.column_stack([np.ones(n), X_sc])
w_gd, losses = gradient_descent_linear(X_sc_b, y, lr=0.05, n_iter=1000)
print(f'\nGradient descent (standardized features):')
print(f'  Final MSE: {losses[-1]:.6f}')
print(f'  Converged at iter ~{next((i for i, l in enumerate(losses) if l < losses[0]*0.01), 1000)}')

# ---- 3. Ridge and Lasso comparison on high-dim data ----
# Simulate p=50 features, n=100, only 3 are truly informative
p = 50
X_hd = rng.standard_normal((100, p))
true_w_sparse = np.zeros(p)
true_w_sparse[[0, 5, 20]] = [3.0, -2.0, 1.5]  # only 3 non-zero features
y_hd = X_hd @ true_w_sparse + rng.normal(0, 0.5, 100)

# Fit OLS, Ridge, Lasso
from numpy.linalg import lstsq
w_ols_hd, _, _, _ = lstsq(X_hd, y_hd, rcond=None)
w_ridge = np.linalg.solve(X_hd.T @ X_hd + 1.0 * np.eye(p), X_hd.T @ y_hd)

lasso = Lasso(alpha=0.1, max_iter=5000)
lasso.fit(X_hd, y_hd)
w_lasso = lasso.coef_

print(f'\nSparse signal recovery (true nonzero at indices 0, 5, 20):')
for name, w in [('OLS', w_ols_hd), ('Ridge', w_ridge), ('Lasso', w_lasso)]:
    nonzero = np.where(np.abs(w) > 0.1)[0]
    print(f'  {name:<8}: nonzero coefs at {nonzero.tolist()}')
    print(f'           |w[0]|={abs(w[0]):.3f}, |w[5]|={abs(w[5]):.3f}, |w[20]|={abs(w[20]):.3f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Gradient descent convergence
ax = axes[0]
ax.plot(losses, 'steelblue', lw=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Training MSE')
ax.set_title('A. Gradient descent convergence')
ax.set_yscale('log')
ax.axhline(losses[-1], color='red', ls='--', lw=0.8, label=f'Final MSE={losses[-1]:.4f}')
ax.legend(fontsize=8)

# Panel B: Actual vs. predicted (OLS)
ax = axes[1]
y_pred_ols = X_b @ w_ols
ax.scatter(y, y_pred_ols, alpha=0.5, s=20, color='steelblue')
lim = [y.min()-0.3, y.max()+0.3]
ax.plot(lim, lim, 'k--', lw=1)
ax.set_xlabel('True y')
ax.set_ylabel('Predicted y')
ax.set_title('B. OLS predictions\n(actual vs. predicted)')
rss = np.sum((y - y_pred_ols)**2)
tss = np.sum((y - y.mean())**2)
r2 = 1 - rss/tss
ax.text(0.05, 0.90, f'R²={r2:.4f}', transform=ax.transAxes, fontsize=9)

# Panel C: Coefficient paths — OLS vs Ridge vs Lasso for sparse problem
ax = axes[2]
ax.bar(np.arange(p) - 0.3, true_w_sparse, width=0.25, color='black', alpha=0.6, label='True')
ax.bar(np.arange(p), w_ridge, width=0.25, color='steelblue', alpha=0.7, label='Ridge')
ax.bar(np.arange(p) + 0.3, w_lasso, width=0.25, color='tomato', alpha=0.7, label='Lasso')
ax.set_xlim(-1, 30)  # show first 30 features
ax.set_xlabel('Feature index')
ax.set_ylabel('Coefficient')
ax.set_title('C. Sparse signal recovery\n(Lasso finds the true nonzero features)')
ax.legend(fontsize=8)
ax.axhline(0, color='grey', lw=0.5)

plt.suptitle('Module 13 NB02: Linear Regression from Scratch', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('linear_regression.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Derive the gradient $\nabla_\mathbf{w} \|\mathbf{y} - X\mathbf{w}\|_2^2$ by hand.
   Verify your result against the code above.
2. Implement Ridge regression from scratch (closed form). Verify your coefficients
   match `sklearn.linear_model.Ridge`.
3. Plot the Ridge coefficient path: vary $\lambda$ from $10^{-3}$ to $10^3$.
   What happens to the coefficients as $\lambda \to \infty$?
4. When would you use Ridge instead of Lasso for a genomics problem? Justify with
   reference to polygenic architecture.

---
## Step 10 — Quiz

1. What are the normal equations? Derive them from the OLS loss.
2. What is the geometric interpretation of OLS?
3. What is the difference between Ridge and Lasso regularization?
4. Why is $X^TX + \lambda I$ always invertible for $\lambda > 0$?
5. Why is OLS not appropriate for RNA-seq count data?

---
## Step 12 — Reflection

> *[In one paragraph: explain why Lasso performs sparse feature selection while Ridge
> does not, using the geometry of the L1 and L2 penalty regions.]*

---
*Next: `03_logistic_regression_from_scratch.ipynb`*